# 🧹 Eksik Veri Yönetimi (Missing Value Handling)

Bu notebook, veri setindeki boşlukları doldurmak için kullanılan gelişmiş şablonları içerir. Yol haritandaki **KNNImputer** ve **IterativeImputer** gibi ileri seviye teknikleri burada bulabilirsin.

In [ ]:
import pandas as pd
import numpy as np

# İlgili kütüphaneler (All_Imports'tan alınmıştır)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

## 1. Basit Doldurma (Simple Imputer)

In [ ]:
def apply_simple_imputer(df_train, df_test, cols, strategy='median'):
    """
    Sayısal sütunlar için ortalama/medyan, kategorik sütunlar için en sık geçen değer.
    Dikkat: Data leakage (Veri sızıntısı) olmaması için sadece train'de fit edilir.
    """
    imputer = SimpleImputer(strategy=strategy)
    
    df_train_imputed = df_train.copy()
    df_test_imputed = df_test.copy()
    
    df_train_imputed[cols] = imputer.fit_transform(df_train[cols])
    df_test_imputed[cols] = imputer.transform(df_test[cols])
    
    return df_train_imputed, df_test_imputed

## 2. Gelişmiş: KNN Imputer (K-En Yakın Komşu ile)
Eksik veriyi, o satıra en çok benzeyen (öklid uzaklığına göre) `n_neighbors` kadar satırın ortalamasını alarak doldurur.

In [ ]:
def apply_knn_imputer(df_train, df_test, cols, n_neighbors=5):
    """
    Özellikle sayısal sütunlarda, o satırın diğer özelliklerine bakarak en benzer evleri/yolcuları bulur
    ve onların ortalamasını eksik değere yazar.
    """
    imputer = KNNImputer(n_neighbors=n_neighbors)
    
    df_train_imputed = df_train.copy()
    df_test_imputed = df_test.copy()
    
    df_train_imputed[cols] = imputer.fit_transform(df_train[cols])
    df_test_imputed[cols] = imputer.transform(df_test[cols])
    
    return df_train_imputed, df_test_imputed

## 3. Profesyonel: Iterative Imputer (MICE)
Eksik sütunu, diğer tüm sütunları kullanarak bir makine öğrenmesi modeli (genelde Ridge veya RandomForest) kurup tahmin ederek doldurur.

In [ ]:
def apply_iterative_imputer(df_train, df_test, cols):
    """
    IterativeImputer (MICE algoritması benzeri). Kaggle'da en çok işe yarayan doldurma tekniklerinden biridir.
    """
    imputer = IterativeImputer(max_iter=10, random_state=42)
    
    df_train_imputed = df_train.copy()
    df_test_imputed = df_test.copy()
    
    df_train_imputed[cols] = imputer.fit_transform(df_train[cols])
    df_test_imputed[cols] = imputer.transform(df_test[cols])
    
    return df_train_imputed, df_test_imputed

## 4. Grup Bazlı Doldurma (Pandas Groupby & Transform)
Bir eksik veriyi genel ortalama ile değil, ait olduğu grubun (örneğin aynı mahalledeki evlerin) ortalamasıyla doldurmak çok daha tutarlıdır.

In [ ]:
# ÖRNEK: House Prices'da 'LotFrontage' sütununu, evin bulunduğu 'Neighborhood' (Mahalle) ortancası ile doldurmak
"""
df_train["LotFrontage"] = df_train.groupby("Neighborhood")["LotFrontage"].transform(
    lambda x: x.fillna(x.median())
)
"""